# ✈️ DS_Day01_22 — Aviation: Why Are Flights Delayed?

**Industry:** Aviation  **Algorithm:** Random Forest

An airline wants to reduce flight delays. This notebook works through each task in the brief:

1. Clean and validate the dataset
2. Exploratory data analysis with descriptive statistics
3. Delay rates by airport
4. Delays by hour and day
5. The effect of aircraft turnaround time
6. Routes that are consistently delayed
7. Operational vs weather-related delays
8. The most significant delay contributors
9. A trained **Random Forest** model and its evaluation
10. **Key insights**, the **4 key visualisations** and a **practical action plan**

**Data provided (schema from the brief):** `Flight_ID, Airport, Route, Aircraft, Scheduled_Departure, Actual_Departure, Weather, Previous_Flight, Turnaround_Time`

> The brief gives only the schema, so the dataset is **synthetic** (see `src/generate_data.py`). The analysis code lives in `src/`, and the Streamlit dashboard (`app.py`) uses the same functions and charts, so both always show the same results.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():  # allow running from a sub-folder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import analysis as A
from src import charts as C
from src import generate_data
from src import report as R
from src.cleaning import DELAY_THRESHOLD, MIN_TURN, clean
from src.model import FEATURES, buffer_scenario, train

# Interactive plotly figures in VS Code / JupyterLab, with an HTML fallback for classic Notebook
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = ROOT / "data" / "raw_flights.csv"
MODELS = ROOT / "models"
REGENERATE_DATA = False   # True -> simulate a fresh raw dataset
RETRAIN_MODEL = False     # True -> re-run the grid search and training (~1 min)

In [2]:
if REGENERATE_DATA or not RAW.exists():
    generate_data.main(RAW)
raw = pd.read_csv(RAW, dtype={"Previous_Flight": "string"})
print(f"Raw rows: {len(raw):,}  |  columns: {list(raw.columns)}")
raw.head(10)

Raw rows: 86,389  |  columns: ['Flight_ID', 'Airport', 'Route', 'Aircraft', 'Scheduled_Departure', 'Actual_Departure', 'Weather', 'Previous_Flight', 'Turnaround_Time']


,Flight_ID,Airport,Route,Aircraft,Scheduled_Departure,Actual_Departure,Weather,Previous_Flight,Turnaround_Time
0,FL27208,DEL,DEL-CCU,A320-104,2025-11-24 20:15,2025-11-24 20:57,Clear,FL27207,80.0
1,FL42344,CCU,CCU-MAA,A320-112,2025-08-15 05:45,2025-08-15 05:51,Cloudy,FL42343,315.0
2,FL15121,HYD,HYD-JAI,B737-101,2025-07-30 09:10,2025-07-30 09:01,Clear,FL15120,50.0
3,FL40698,BOM,BOM-COK,A320-111,2025-10-16 22:05,2025-10-16 22:10,Clear,FL40697,35.0
4,FL54164,MAA,MAA - DEL,ATR72-102,2025-10-21 19:05,2025-10-21 19:23,Thunderstorm,FL54163,45.0
5,FL60492,MAA,MAA-DEL,A320-117,2025-01-18 11:45,2025-01-18 11:54,Haze,FL60491,40.0
6,FL62519,HYD,HYD-DEL,A321-103,2025-02-09 08:40,2025-02-09 08:38,Clear,FL62518,75.0
7,FL13436,CCU,CCU-DEL,ATR72-101,2025-09-21 19:45,2025-09-21 20:36,Cloudy,FL13435,15.0
8,FL80020,CCU,CCU-HYD,A320-123,2025-02-24 11:15,2025-02-24 11:13,Cloudy,FL80019,50.0
9,FL45504,GOI,GOI-DEL,A321-101,2025-04-22 17:55,2025-04-22 18:11,Haze,FL45503,105.0


In [3]:
# Quick data-quality profile of the raw file
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
print("Exact duplicate rows:", raw.duplicated().sum())
print("Airport labels:", sorted(raw["Airport"].dropna().unique())[:30])
print("Weather labels:", sorted(raw["Weather"].dropna().unique()))
print("Turnaround_Time < 0:", (raw["Turnaround_Time"] < 0).sum(), "| > 1440:", (raw["Turnaround_Time"] > 1440).sum())
profile

Exact duplicate rows: 1276
Airport labels: [' AMD ', ' BLR ', ' BOM ', ' CCU ', ' COK ', ' DEL ', ' GOI ', ' HYD ', ' JAI ', ' LKO ', ' MAA ', ' PNQ ', ' blr ', ' bom ', ' ccu ', ' del ', ' hyd ', ' maa ', 'AMD', 'BLR', 'BOM', 'CCU', 'COK', 'DEL', 'GOI', 'HYD', 'JAI', 'LKO', 'MAA', 'PNQ']
Weather labels: ['CLEAR', 'CLOUDY', 'Clear', 'Cloudy', 'FOG', 'Fog', 'HAZE', 'Haze', 'RAIN', 'Rain', 'THUNDERSTORM', 'Thunderstorm', 'clear ', 'cloudy ', 'fog ', 'haze ', 'rain ', 'thunderstorm ']
Turnaround_Time < 0: 251 | > 1440: 172


,dtype,missing,missing_%,unique
Flight_ID,str,0,0.00,85113
Airport,str,0,0.00,42
Route,str,0,0.00,199
Aircraft,str,0,0.00,44
Scheduled_Departure,str,0,0.00,51359
Actual_Departure,str,689,0.80,75950
Weather,str,2590,3.00,18
Previous_Flight,string,44,0.05,85069
Turnaround_Time,float64,1335,1.55,168


## 2. Cleaning and validation

`src/cleaning.py::clean` applies these steps and logs how many rows each one affected:

| Issue | Treatment |
|---|---|
| Exact duplicate rows | dropped |
| Inconsistent case and whitespace in Airport / Route / Weather | trimmed and normalised |
| `Airport` ≠ origin in `Route` | `Airport` set from the route, since the route is the more reliable field |
| Missing weather | labelled `Unknown` (kept as its own category rather than guessed) |
| Mixed timestamp formats | ISO format parsed first, then DD/MM/YYYY as a fallback |
| Missing `Actual_Departure` | flagged as cancelled or not recorded, and left out of delay metrics |
| Actual departure logged one day early | corrected by adding 1 day |
| Negative turnaround (sign error) | absolute value taken |
| Turnaround > 24 h, or missing | median for the aircraft type × airport (same-day turn) or the overnight median |

**Engineered features:**
- `Delay_Min`, and `Is_Delayed` (delay > 15 min, the industry on-time standard)
- hour, day and month of departure
- `Aircraft_Type`, with its minimum ground time `Min_Turn`
- `Turn_Slack` = turnaround − `Min_Turn`
- `Prev_Delay_Min`: the previous leg's delay, joined via `Previous_Flight`
- `Propagated_Min`: the part of that inbound delay the slack could not absorb
- `Delay_Cause`: the primary cause assigned to each delay

In [4]:
df, log = clean(raw)
log

,Step,Rows affected,Action
0,Raw rows loaded,86389,-
1,Exact duplicate rows,1276,Dropped
2,Duplicate Flight_ID (non-identical),0,Kept first occurrence
3,Inconsistent text in Airport,2524,Trimmed / case-normalised
4,Inconsistent text in Route,1273,Trimmed / case-normalised
5,Inconsistent text in Aircraft,0,Trimmed / case-normalised
6,Inconsistent text in Weather,2464,Trimmed / case-normalised
7,Airport does not match Route origin,238,Airport set to Route origin
8,Unrecognised Weather label,0,Set to missing
9,Missing Weather,2553,Labelled 'Unknown' (kept as its own category)


In [5]:
# Validation checks on the cleaned data
v = A.valid(df)
checks = {
    "No duplicate Flight_IDs": df["Flight_ID"].is_unique,
    "Airport always equals Route origin": (df["Airport"] == df["Origin"]).all(),
    "All weather labels valid": df["Weather"].isin(["Clear", "Cloudy", "Haze", "Rain", "Thunderstorm", "Fog", "Unknown"]).all(),
    "No negative turnaround": (df["Turnaround_Time"] >= 0).all(),
    "No missing turnaround": df["Turnaround_Time"].notna().all(),
    "Delays within [-60 min, 24 h]": v["Delay_Min"].between(-60, 1440).all(),
    "Is_Delayed consistent with Delay_Min": ((v["Delay_Min"] > DELAY_THRESHOLD) == (v["Is_Delayed"] == 1)).all(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values())
print(f"\nClean flights: {len(df):,}  |  with valid delay: {len(v):,}")
df.head()

PASS No duplicate Flight_IDs
PASS Airport always equals Route origin
PASS All weather labels valid
PASS No negative turnaround
PASS No missing turnaround
PASS Delays within [-60 min, 24 h]
PASS Is_Delayed consistent with Delay_Min

Clean flights: 85,113  |  with valid delay: 84,433


,Flight_ID,Airport,Route,Aircraft,Scheduled_Departure,Actual_Departure,Weather,Previous_Flight,Turnaround_Time,Origin,Destination,Delay_Min,Cancelled_or_Missing,Aircraft_Type,Min_Turn,Turnaround_Imputed,Is_Delayed,Date,Dep_Hour,Day_of_Week,Month,Month_Name,Prev_Delay_Min,Prev_Delay_Known,Turn_Slack,Turn_Bucket,Short_Turnaround,Adverse_Weather,Propagated_Min,Delay_Cause,Cause_Group
0,FL27405,BOM,BOM-DEL,A320-105,2025-01-01 05:30:00,2025-01-01 05:33:00,Clear,<NA>,600.0,BOM,DEL,3.0,False,A320,35,False,0,2025-01-01,5,Wed,1,Jan,0.0,False,600.0,Overnight,False,False,0.0,On time,On time
1,FL14001,HYD,HYD-BOM,B737-101,2025-01-01 05:30:00,2025-01-01 05:33:00,Clear,<NA>,600.0,HYD,BOM,3.0,False,B737,40,False,0,2025-01-01,5,Wed,1,Jan,0.0,False,600.0,Overnight,False,False,0.0,On time,On time
2,FL19741,BOM,BOM-AMD,A320-103,2025-01-01 05:30:00,2025-01-01 05:28:00,Clear,<NA>,600.0,BOM,AMD,-2.0,False,A320,35,False,0,2025-01-01,5,Wed,1,Jan,0.0,False,600.0,Overnight,False,False,0.0,On time,On time
3,FL52530,DEL,DEL-GOI,ATR72-102,2025-01-01 05:35:00,2025-01-01 06:25:00,Fog,<NA>,600.0,DEL,GOI,50.0,False,ATR72,25,False,1,2025-01-01,5,Wed,1,Jan,0.0,False,600.0,Overnight,False,True,0.0,Weather,Weather
4,FL93186,BOM,BOM-DEL,A320-126,2025-01-01 05:35:00,2025-01-01 05:39:00,Cloudy,<NA>,600.0,BOM,DEL,4.0,False,A320,35,False,0,2025-01-01,5,Wed,1,Jan,0.0,False,600.0,Overnight,False,False,0.0,On time,On time


## 3. Exploratory data analysis: descriptive statistics

In [6]:
K = A.kpis(df)
display(Markdown(
    f"- **Flights analysed:** {K['valid']:,}\n"
    f"- **On-time performance (≤{DELAY_THRESHOLD} min):** {K['otp']:.1%}\n"
    f"- **Delay rate:** {K['delay_rate']:.1%}\n"
    f"- **Mean departure delay (early = 0):** {K['avg_delay']:.1f} min\n"
    f"- **Median delay of delayed flights:** {K['median_delay_delayed']:.0f} min\n"
    f"- **Total delay hours (delayed flights):** {K['delay_minutes'] / 60:,.0f}"
))
A.descriptive_stats(df)

- **Flights analysed:** 84,433
- **On-time performance (≤15 min):** 72.7%
- **Delay rate:** 27.3%
- **Mean departure delay (early = 0):** 13.0 min
- **Median delay of delayed flights:** 28 min
- **Total delay hours (delayed flights):** 14,303

,count,mean,std,min,25%,50%,75%,90%,95%,max,skew
Delay_Min,84433.0,11.84,22.10,-10.0,-1.0,6.0,17.0,35.0,50.0,528.0,3.85
Turnaround_Time,84433.0,116.05,141.77,15.0,40.0,50.0,80.0,400.0,455.0,620.0,1.77
Turn_Slack,84433.0,79.57,141.72,-15.0,5.0,15.0,45.0,360.0,415.0,600.0,1.77
Prev_Delay_Min,84433.0,12.87,21.21,0.0,0.0,5.0,17.0,34.0,50.0,528.0,4.25
Dep_Hour,84433.0,13.42,5.22,5.0,9.0,13.0,18.0,21.0,22.0,22.0,0.02


In [7]:
# Categorical composition
for col in ["Airport", "Aircraft_Type", "Weather"]:
    display(v[col].value_counts().rename("flights").to_frame()
            .assign(share=lambda d: (d["flights"] / d["flights"].sum()).round(3)).T)

Airport,DEL,BOM,BLR,HYD,MAA,CCU,GOI,AMD,LKO,COK,JAI,PNQ
flights,13421,13402,13393,11421,10801,10133,2012,1996,1979,1974,1971,1930
share,0.159,0.159,0.159,0.135,0.128,0.12,0.024,0.024,0.023,0.023,0.023,0.023


Aircraft_Type,A320,B737,A321,ATR72
flights,50231.000,18905.000,9235.000,6062.000
share,0.595,0.224,0.109,0.072


Weather,Clear,Cloudy,Rain,Haze,Unknown,Thunderstorm,Fog
flights,45385,19105,10175,4368,2536,2224,640
share,0.538,0.226,0.121,0.052,0.03,0.026,0.008


In [8]:
C.delay_histogram(df).show()
C.monthly_rate(df).show()

The delay distribution is **strongly right-skewed** (skew > 2). Most flights leave close to schedule, but a long tail runs past 2 hours. That's why we report the **delay rate** and medians or P90 alongside means.

## 4. Delay rates by airport

Delay rates are shown with 95% Wilson confidence intervals.

In [9]:
air = A.by_airport(df)
display(air.style.format({"Delay_Rate": "{:.1%}", "CI_Low": "{:.1%}", "CI_High": "{:.1%}",
                          "Avg_Delay": "{:.1f}", "P90_Delay": "{:.0f}", "Weather_Share": "{:.1%}"}))
C.airport_rates(df).show()

,Airport,Flights,Delayed,Avg_Delay,P90_Delay,Weather_Share,Delay_Rate,CI_Low,CI_High
0,BOM,13402,5038,17.3,44,34.1%,37.6%,36.8%,38.4%
1,DEL,13421,4550,15.9,41,23.7%,33.9%,33.1%,34.7%
2,CCU,10133,2823,13.1,34,33.3%,27.9%,27.0%,28.7%
3,BLR,13393,3413,12.0,32,30.1%,25.5%,24.8%,26.2%
4,GOI,2012,501,12.3,34,43.5%,24.9%,23.1%,26.8%
5,PNQ,1930,438,10.9,30,41.6%,22.7%,20.9%,24.6%
6,COK,1974,447,11.3,31,38.3%,22.6%,20.9%,24.5%
7,LKO,1979,442,11.8,32,31.0%,22.3%,20.6%,24.2%
8,AMD,1996,426,11.2,33,35.2%,21.3%,19.6%,23.2%
9,MAA,10801,2296,10.5,29,31.5%,21.3%,20.5%,22.0%


## 5. Delays by hour and day

In [10]:
C.hour_day_heatmap(df, "rate").show()
C.hour_day_heatmap(df, "mean").show()

In [11]:
C.hour_rates(df).show()
C.day_rates(df).show()
display(A.by_day(df).style.format({"Delay_Rate": "{:.1%}", "Avg_Delay": "{:.1f}"}))

,Day_of_Week,Flights,Delay_Rate,Avg_Delay
0,Mon,12007,26.7%,11.3
1,Tue,12017,24.6%,11.1
2,Wed,12205,25.5%,11.4
3,Thu,12123,26.5%,11.2
4,Fri,12003,31.8%,13.9
5,Sat,12022,24.5%,10.2
6,Sun,12056,31.2%,13.7


## 6. Effect of aircraft turnaround time

**Turnaround slack** is the scheduled ground time minus the minimum ground time for the aircraft type (A320 35, A321 45, B737 40, ATR72 25 min). A negative slack means the turnaround is scheduled *below* what the aircraft physically needs.

In [12]:
ts = A.turnaround_stats(df)
display(Markdown(
    f"- Same-day turns scheduled **below the type minimum**: {ts['short_share']:.1%}\n"
    f"- Delay rate: **{ts['short_rate']:.1%}** below the minimum vs **{ts['ok_rate']:.1%}** otherwise "
    f"(χ² = {ts['chi2']:,.0f}, p = {ts['chi2_p']:.1e})\n"
    f"- Spearman ρ (slack vs delay, same-day turns): **{ts['spearman_rho']:.2f}** (p = {ts['spearman_p']:.1e})\n"
    f"- Pearson r (inbound delay vs outbound delay): **{ts['prop_corr']:.2f}**"
))
display(A.turnaround_effect(df).style.format({"Delay_Rate": "{:.1%}", "Avg_Delay": "{:.1f}", "Avg_Prev_Delay": "{:.1f}"}))
C.turnaround_rates(df).show()

- Same-day turns scheduled **below the type minimum**: 13.0%
- Delay rate: **69.7%** below the minimum vs **24.8%** otherwise (χ² = 7,485, p = 0.0e+00)
- Spearman ρ (slack vs delay, same-day turns): **-0.29** (p = 0.0e+00)
- Pearson r (inbound delay vs outbound delay): **0.46**

,Turn_Bucket,Flights,Delay_Rate,Avg_Delay,Avg_Prev_Delay
0,Below minimum,9067,69.7%,30.1,12.3
1,0-10 min slack,17727,30.5%,13.4,11.8
2,10-20 min slack,20265,25.2%,11.1,11.9
3,20-40 min slack,13778,21.3%,9.4,11.5
4,40+ min slack,8859,17.8%,7.4,12.0
5,Overnight,14737,11.4%,4.7,17.6


In [13]:
C.inbound_x_slack_heatmap(df).show()
C.propagation_scatter(df).show()
C.leg_accumulation(df).show()

**Reading the charts:** in the heatmap, each row's delay rate falls as slack grows, because ground time absorbs a late inbound aircraft. Down the rows, it rises sharply unless there is plenty of slack. The scatter shows many flights sitting close to the *y = x* line, meaning the inbound delay was passed on in full. The last chart shows delays **building up with each leg** of an aircraft's day. Together these show how knock-on (reactionary) delay works.

## 7. Do certain routes consistently experience delays?

A route counts as **consistently delayed** when:
1. its delay rate beats the network's monthly rate in **at least 10 of 12 months**, and
2. a one-sided **binomial test** against the network rate gives **p < 0.01**.

It is **route-specific** if it is also at least 15% worse than its *own departure airport*. That separates a problem with the route itself from general congestion at the hub.

In [14]:
routes = A.route_consistency(df)
cols = ["Route", "Flights", "Delay_Rate", "Avg_Delay", "Months_Above_Network", "Monthly_Std", "p_value",
        "Lift_vs_Network", "Lift_vs_Origin", "Consistently_Delayed", "Route_Specific"]
display(routes[cols].head(20).style.format({"Delay_Rate": "{:.1%}", "Avg_Delay": "{:.1f}", "Monthly_Std": "{:.3f}",
                                            "p_value": "{:.1e}", "Lift_vs_Network": "{:.2f}", "Lift_vs_Origin": "{:.2f}"}))
print("Consistently delayed:", routes.loc[routes["Consistently_Delayed"], "Route"].tolist())
print("Route-specific:      ", routes.loc[routes["Route_Specific"], "Route"].tolist())

,Route,Flights,Delay_Rate,Avg_Delay,Months_Above_Network,Monthly_Std,p_value,Lift_vs_Network,Lift_vs_Origin,Consistently_Delayed,Route_Specific
0,BOM-DEL,1977,63.1%,28.2,12,0.060,1.0e-241,2.31,1.68,True,True
1,DEL-BOM,2021,57.7%,25.1,12,0.086,2.6e-181,2.12,1.70,True,True
2,BOM-GOI,643,55.1%,23.4,12,0.103,1.0e-49,2.02,1.46,True,True
3,CCU-DEL,2052,49.6%,20.8,12,0.058,9.8e-102,1.82,1.78,True,True
4,DEL-LKO,645,45.0%,20.3,12,0.092,5.4e-22,1.65,1.33,True,True
5,BLR-BOM,2022,41.7%,17.7,12,0.080,9.4e-45,1.53,1.64,True,True
6,BOM-LKO,703,34.4%,15.0,11,0.076,1.8e-05,1.26,0.92,True,False
7,BOM-CCU,1971,33.4%,14.5,11,0.086,9.4e-10,1.23,0.89,True,False
8,BOM-BLR,2081,32.5%,14.3,11,0.091,8.2e-08,1.19,0.86,True,False
9,BOM-JAI,617,32.1%,15.5,9,0.103,4.5e-03,1.18,0.85,False,False


Consistently delayed: ['BOM-DEL', 'DEL-BOM', 'BOM-GOI', 'CCU-DEL', 'DEL-LKO', 'BLR-BOM', 'BOM-LKO', 'BOM-CCU', 'BOM-BLR', 'BOM-HYD']
Route-specific:       ['BOM-DEL', 'DEL-BOM', 'BOM-GOI', 'CCU-DEL', 'DEL-LKO', 'BLR-BOM']


In [15]:
C.top_routes(df, routes).show()
C.route_month_heatmap(df, routes["Route"].head(12).tolist()).show()

## 8. Operational vs weather-related delays

The data has no cause code, so each delayed departure (> 15 min) gets one **primary cause**. The rules are checked in this order:

1. **Weather**: Fog, Thunderstorm or Rain at the origin airport
2. **Reactionary (knock-on)**: the inbound delay left after the slack accounts for at least 50% of the delay
3. **Short turnaround**: ground time scheduled below the type minimum
4. **Airport / other operational**: congestion, air traffic control, handling, crew or technical

Causes 2–4 are **operational**, meaning the airline or airport can control them.

In [16]:
causes = A.cause_breakdown(df)
display(causes.style.format({"Delay_Minutes": "{:,.0f}", "Avg_Delay": "{:.1f}",
                             "Share_Flights": "{:.1%}", "Share_Minutes": "{:.1%}"}))
display(causes.groupby("Cause_Group")[["Flights", "Delay_Minutes"]].sum()
        .assign(Share_Minutes=lambda d: d["Delay_Minutes"] / d["Delay_Minutes"].sum())
        .style.format({"Delay_Minutes": "{:,.0f}", "Share_Minutes": "{:.1%}"}))
C.cause_hours(df).show()

,Cause_Group,Delay_Cause,Flights,Delay_Minutes,Avg_Delay,Share_Flights,Share_Minutes
3,Weather,Weather,7106,"319,954",45.0,30.9%,37.3%
1,Operational,Reactionary (knock-on),6585,"266,607",40.5,28.6%,31.1%
0,Operational,Airport / other operational,6070,"164,743",27.1,26.4%,19.2%
2,Operational,Short turnaround,3255,"106,893",32.8,14.1%,12.5%


,Flights,Delay_Minutes,Share_Minutes
Cause_Group,,,
Operational,15910,"538,243",62.7%
Weather,7106,"319,954",37.3%


In [17]:
wt = A.weather_test(df)
display(Markdown(
    f"Median delay in adverse weather: **{wt['adverse_median']:.0f} min** vs **{wt['fair_median']:.0f} min** in "
    f"clear or cloudy conditions. A one-sided Mann-Whitney U test gives p = {wt['mw_p']:.1e}. "
    f"Adverse weather affects {wt['adverse_share']:.1%} of departures."
))
display(A.by_weather(df).style.format({"Delay_Rate": "{:.1%}", "Avg_Delay": "{:.1f}", "P90_Delay": "{:.0f}",
                                       "Excess_vs_Clear": "{:+.1f}"}))
C.weather_delay(df).show()
C.monthly_cause_split(df).show()

Median delay in adverse weather: **18 min** vs **4 min** in clear or cloudy conditions. A one-sided Mann-Whitney U test gives p = 0.0e+00. Adverse weather affects 15.4% of departures.

,Weather,Flights,Delay_Rate,Avg_Delay,P90_Delay,Excess_vs_Clear
2,Fog,640,69.8%,55.1,131,+47.1
5,Thunderstorm,2224,72.1%,48.3,114,+40.4
4,Rain,10175,49.7%,20.7,48,+12.7
3,Haze,4368,36.3%,15.4,38,+7.4
6,Unknown,2536,26.8%,11.8,33,+3.8
1,Cloudy,19105,22.9%,9.8,29,+1.8
0,Clear,45385,20.4%,8.0,27,+0.0


## 9. Random Forest model

**Task:** predict whether a departure will be delayed by more than 15 minutes (a binary classification).

**Features:** only information known *before push-back*, so nothing leaks from the actual departure time: departure airport, destination, aircraft type, weather, scheduled hour, day of week, month, planned turnaround, turnaround slack and inbound (previous-leg) delay.

**Set-up:**
- Categorical features are ordinal-encoded.
- `class_weight='balanced_subsample'` handles the imbalance (about 27% of flights are delayed).
- A stratified 80/20 train/test split is used.
- The grid search tries `max_depth` ∈ {12, 20} and `min_samples_leaf` ∈ {5, 20}, scored by 3-fold stratified CV ROC-AUC.
- Final results come from the 20% test set that training never saw.

In [18]:
if RETRAIN_MODEL or not (MODELS / "rf_model.joblib").exists():
    metrics = train(df, MODELS)
else:
    metrics = json.loads((MODELS / "metrics.json").read_text())
model = joblib.load(MODELS / "rf_model.joblib")
print("Features:", FEATURES)
print("Best hyper-parameters:", metrics["best_params"], "| CV ROC-AUC:", round(metrics["cv_auc"], 3))
display(pd.DataFrame(metrics["cv_results"]))

Features: ['Airport', 'Destination', 'Aircraft_Type', 'Weather', 'Dep_Hour', 'Day_Num', 'Month', 'Turnaround_Time', 'Turn_Slack', 'Prev_Delay_Min']
Best hyper-parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 20} | CV ROC-AUC: 0.874


,params,mean_test_score,std_test_score
0,"{'rf__max_depth': 12, 'rf__max_features': 'sqr...",0.872982,0.001362
1,"{'rf__max_depth': 12, 'rf__max_features': 'sqr...",0.872307,0.001267
2,"{'rf__max_depth': 20, 'rf__max_features': 'sqr...",0.873894,0.001767
3,"{'rf__max_depth': 20, 'rf__max_features': 'sqr...",0.874186,0.001380


In [19]:
eval_table = pd.DataFrame({
    "Metric": ["Accuracy", "Majority-class baseline accuracy", "Precision (delayed)", "Recall (delayed)",
               "F1 (delayed)", "ROC-AUC", "PR-AUC (average precision)", "Positive-class base rate"],
    "Value": [metrics["accuracy"], metrics["baseline_accuracy"], metrics["precision"], metrics["recall"],
              metrics["f1"], metrics["roc_auc"], metrics["pr_auc"], metrics["positive_rate"]],
})
display(eval_table.style.format({"Value": "{:.3f}"}).hide(axis="index"))
print(f"Train rows: {metrics['n_train']:,}  |  Test rows: {metrics['n_test']:,}")

Metric,Value
Accuracy,0.801
Majority-class baseline accuracy,0.727
Precision (delayed),0.607
Recall (delayed),0.763
F1 (delayed),0.676
ROC-AUC,0.879
PR-AUC (average precision),0.773
Positive-class base rate,0.273


Train rows: 67,546  |  Test rows: 16,887


In [20]:
C.confusion(metrics).show()
C.roc(metrics).show()
C.precision_recall(metrics).show()

### Most significant delay contributors

**Permutation importance** measures how much the test ROC-AUC drops when a feature's values are shuffled. It is more reliable than the forest's built-in (impurity-based) importance, which tends to favour numeric features with many distinct values.

In [21]:
C.permutation_importance(metrics).show()
C.impurity_importance(metrics).show()
pd.DataFrame({"Permutation (ROC-AUC drop)": metrics["perm_importance"], "Impurity (Gini)": metrics["importance"]}) \
  .rename(index=C.FEATURE_LABELS).sort_values("Permutation (ROC-AUC drop)", ascending=False)

,Permutation (ROC-AUC drop),Impurity (Gini)
Inbound (previous-leg) delay,0.121907,0.254812
Turnaround slack,0.082367,0.190300
Weather,0.078585,0.182513
Departure airport,0.035629,0.068142
Destination,0.017122,0.048852
Turnaround time,0.013517,0.126580
Departure hour,0.009453,0.068015
Day of week,0.001706,0.019642
Aircraft type,0.001064,0.012295
Month,0.000994,0.028850


In [22]:
F = R.facts(df, metrics)
C.tail_rates(df, set(F["problem_tails"]["Aircraft"])).show()
F["problem_tails"]

,Aircraft,Aircraft_Type,Flights,Delay_Rate,Own_Delay,Avg_Delay,Z
23,A320-124,A320,1929,0.664075,0.247278,28.611198,38.61255
36,B737-103,B737,1891,0.657324,0.243258,28.872554,37.571113


### What-if: enforcing a minimum turnaround buffer

We raise every same-day turnaround to at least *N* minutes of slack, score all flights again with the Random Forest, and count how many delayed departures the model predicts. Inbound delays are held fixed, so the knock-on benefit to later legs isn't counted and the estimate is conservative. This is the model's estimate, not proof of cause and effect.

In [23]:
rows = []
for s in range(0, 35, 5):
    sc = buffer_scenario(model, df, s)
    rows.append({"Min_Slack": s, "Turns_Retimed": sc["flights_changed"], "Ground_Hours_Added": sc["minutes_added"] / 60,
                 "Predicted_Delayed": sc["new"], "Avoided": sc["avoided"], "Reduction": sc["rel"],
                 "OTP_Gain": sc["avoided"] / K["valid"]})
scen = pd.DataFrame(rows)
display(scen.style.format({"Turns_Retimed": "{:,}", "Ground_Hours_Added": "{:,.0f}", "Predicted_Delayed": "{:,.0f}",
                           "Avoided": "{:,.0f}", "Reduction": "{:.1%}", "OTP_Gain": "{:+.1%}"}).hide(axis="index"))
C.buffer_curve(scen).show()

Min_Slack,Turns_Retimed,Ground_Hours_Added,Predicted_Delayed,Avoided,Reduction,OTP_Gain
0,"9,067","1,329","20,674","2,342",10.2%,+2.8%
5,"17,287","2,769","20,266","2,750",11.9%,+3.3%
10,"26,794","5,002","19,754","3,262",14.2%,+3.9%
15,"38,295","8,193","19,247","3,769",16.4%,+4.5%
20,"47,059","12,115","18,456","4,560",19.8%,+5.4%
25,"54,309","16,641","18,420","4,596",20.0%,+5.4%
30,"54,309","21,166","18,123","4,893",21.3%,+5.8%


## 10. Key insights

These are computed from the data, so they update automatically if the dataset is regenerated.

In [24]:
display(Markdown("\n\n".join(f"**{i}. {t}.** {b}" for i, (t, b) in enumerate(R.insights(F), 1))))

**1. Delays concentrate at two hubs.** Network on-time performance is **72.7%** (27.3% of departures delayed >15 min). **BOM** (38%) and **DEL** (34%) run the worst delay rates and together generate **42%** of all delayed departures from 32% of flights; HYD is best at 20%.

**2. Delays build up through the day.** Departures between 17:00 and 22:59 are delayed **37%** of the time vs **12%** for the first wave (05:00–07:59) — a **3.1×** gap. Fri (32%) and Sun (31%) are the worst days; Sat is best (24%).

**3. Tight turnarounds are the single most controllable cause.** **13%** of same-day turns are scheduled *below* the aircraft type's minimum ground time; these depart late **70%** of the time vs **25%** for the rest (χ² p < 0.001). Delay rate falls steadily with slack — from 30% (0–10 min) to 18% (40+ min).

**4. Knock-on delay is the largest operational contributor.** Reactionary delay — a late inbound aircraft that the ground time cannot absorb — explains **31%** of delay minutes. Inbound and outbound delays correlate at r = 0.46, and the Random Forest ranks *Inbound (previous-leg) delay* as its most important predictor.

**5. A few routes are late every single month.** **10** routes beat the network delay rate in ≥10 of 12 months (binomial p < 0.01). **6** are worse than their own departure airport by ≥15% — a route-specific problem, not just hub congestion: BOM-DEL (63%), DEL-BOM (58%), BOM-GOI (55%), CCU-DEL (50%), DEL-LKO (45%), BLR-BOM (42%).

**6. Weather matters, but operations matter more.** Weather-attributed delays are **37%** of delay minutes; **63%** are operational. Fog and thunderstorms add **+47** and **+40 min** average delay vs clear skies, but affect only 3.4% of departures — concentrated in the monsoon (Jun–Sep) and winter fog (Dec–Jan) months.

**7. 2 aircraft are chronic under-performers.** Tails **A320-124, B737-103** run delay rates of 66%, 66% vs a fleet average of 27% (≥1.5× fleet, z > 4) — a reliability / maintenance signal rather than a scheduling one.

## 11. The four key visualisations

In [25]:
C.airport_rates(df).show()        # 1. Delay rate by airport
C.hour_day_heatmap(df).show()     # 2. Delays by hour and day
C.turnaround_rates(df).show()     # 3. Effect of turnaround time
C.cause_hours(df).show()          # 4. Weather vs operational delay contributors

## 12. Practical action plan

In [26]:
plan = R.action_plan(F, buffer_scenario(model, df, 10))
with pd.option_context("display.max_colwidth", None):
    display(plan.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

#,Action,Evidence,KPI / target,Owner,Horizon
1,Re-time schedules so no turnaround is below the aircraft-type minimum; add a 10-min buffer on hub turns after 15:00,13% of turns below minimum → 70% delayed vs 25%,Below-minimum turns → 0%; OTP +3.9% (model estimate),Network planning,Next schedule season
2,Contain knock-on delay: hold spare aircraft/crew at BOM and DEL in the evening bank; swap tails when inbound delay exceeds available slack,Reactionary = 31% of delay minutes; 'Inbound (previous-leg) delay' is the top model feature,Reactionary delay minutes −25%,Ops control centre (OCC),0–3 months
3,"Hub recovery plan at BOM and DEL: dedicated turn coordinators, priority ground-handling for departures, slot review with ATC",42% of delayed departures from 32% of flights,Hub delay rate to network average,Airport ops / ground handling,0–6 months
4,"Fix the route-specific problem routes: add block/turn time, move to less congested slots, or reassign reliable aircraft: BOM-DEL, DEL-BOM, BOM-GOI, CCU-DEL, DEL-LKO, BLR-BOM",6 routes worse than network in ≥10/12 months and ≥15% worse than their origin airport,Each route within 1.15× origin delay rate,Network planning,Next schedule season
5,"Protect the evening bank (17:00–22:59) and Fri/Sun peaks: spread departures, add standby staff, avoid scheduling tight turns late in the aircraft's day",Evening delay rate 37% vs 12% first wave; Fri/Sun worst days,Evening delay rate −10 pts,Network planning / HR rostering,3–6 months
6,Seasonal weather playbooks: pre-emptive re-timing and CAT-III-capable crew/aircraft for north-India fog (Dec–Jan); monsoon buffers at BOM/GOI/COK (Jun–Sep),"Weather = 37% of delay minutes; fog +47 min, thunderstorm +40 min vs clear",Weather delay minutes −15% in season,OCC / flight ops,Before next season
7,"Reliability review of chronic tails A320-124, B737-103 (deferred defects, maintenance findings); deploy the RF risk score daily so the OCC sees high-risk departures before they happen",Tail delay rate z > 4 vs fleet; RF ROC-AUC 0.88,Tail delay rates within fleet ±5 pts,Engineering & maintenance / OCC,0–3 months


### Summary

- **Where:** the BOM and DEL hubs, the evening bank, Fridays and Sundays, and a small set of routes that are late every month.
- **Why:** mostly **operational** causes: knock-on delay and turnarounds scheduled below the minimum ground time. Weather is a large but seasonal and partly unavoidable factor.
- **Model:** the Random Forest ranks well above chance (ROC-AUC ≈ 0.88) and confirms that inbound delay and turnaround slack are the biggest drivers.
- **What to do first:** fix turnaround times in the schedule. This is the most controllable lever, and the model estimates a double-digit % reduction in delayed departures from a 10-minute buffer. Then contain knock-on delay at the hubs, fix the route-specific problems, prepare seasonal weather playbooks and review the unreliable aircraft.